# Milestone 4 continuation-batch orchestrator

Status: **PREPARED_FOR_KAGGLE_NO_GPU_RESULTS**. Use a brand-new T4 x2 allocation with Internet enabled. Configure private Kaggle secrets `HF_TOKEN` and `M4_BATCH_ID`. This notebook schedules unchanged logical shards sequentially; every serving cell still starts a fresh vLLM server.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

EXPECTED_SOURCE_COMMIT = 'cf0ce4f85a0753380468c48856ddb2579d31cf19'
BATCH_NOTEBOOK_SOURCE_DIGEST = '25d857d7df5a0ab4bb54735d852f293865f5bf3d0419d7fb612204775599a297'
EXPECTED_FILES = {
    'scripts/kaggle_m4_execute_batch.py': 'd33bf59464a84b143806c859b464f39fbaff07d6850169399e1275fd38531d10',
    'scripts/kaggle_m4_multimodel_crossover.py': '70e8b21d70322fcfb409ccb951f0d540b185c793f03a56b2ba4f9095e3ed948f',
    'research/M4_EXECUTION_PLAN.json': 'd97c8ed1ec8eec7e61e61c617b50d5d48287a253aa28f1e7834b0694fa419d8b',
    'research/model_matrix.json': '7e6eae2f674c394d5db262a02ddd601dfadb17807c4c62e6a252f3e9656664c4',
    'research/m4_protocol.json': '0bb8c6d15b8a6f3d7acc0d1716f99ab34bb4f97ed0e38dd1e26b8f1c92fe99e3',
    'research/M4_BATCH_EXECUTION_PLAN_V2.json': 'dc637d8e98a0478833716617e64026a91791aa4518906381d388db3fbe762461',
    'research/M4_BATCH_PROTOCOL_AMENDMENT_V2.md': '6194397fa1a9f6a80cf6a0df45f689667c75d36644cbbaa2e9d55ac8a4ec8a55',
}
WORK = Path('/kaggle/working'); SOURCE = WORK/'kaggle-vllm-source'; RUNTIME = WORK/'kaggle-vllm-runtime'; CACHE = WORK/'kaggle-vllm-cache'; OUTPUT_ROOT = WORK/'m4-batch-work'; HF_HOME = WORK/'hf-cache'
assert Path('/kaggle').is_dir() and not SOURCE.exists() and not RUNTIME.exists() and not OUTPUT_ROOT.exists(), 'Use a fresh Kaggle allocation'
secrets = UserSecretsClient(); token = secrets.get_secret('HF_TOKEN'); batch_id = secrets.get_secret('M4_BATCH_ID')
assert token and batch_id, 'Configure HF_TOKEN and M4_BATCH_ID Kaggle secrets'; os.environ['HF_TOKEN'] = token; os.environ['HF_HOME'] = str(HF_HOME)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'kaggle-vllm[hub]==0.2.0'], check=True)
subprocess.run(['git', 'init', str(SOURCE)], check=True); subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', 'https://github.com/kaggle-vllm/kaggle-vllm.git'], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', EXPECTED_SOURCE_COMMIT], check=True); subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
head = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip(); dirty = subprocess.check_output(['git', '-C', str(SOURCE), 'status', '--porcelain'], text=True).strip(); assert head == EXPECTED_SOURCE_COMMIT and not dirty
for relative, expected in EXPECTED_FILES.items():
    actual = hashlib.sha256((SOURCE/relative).read_bytes()).hexdigest(); assert actual == expected, f'source digest mismatch: {relative}'
plan = json.loads((SOURCE/'research/M4_BATCH_EXECUTION_PLAN_V2.json').read_text()); matches = [item for item in plan['batches'] if item['batch_id'] == batch_id]; assert len(matches) == 1, 'M4_BATCH_ID is not frozen'; batch = matches[0]
disk = shutil.disk_usage(WORK); reserve = plan['resource_policy']['disk_safety_reserve_bytes']; assert disk.free > reserve, 'disk reserve unavailable before bootstrap'
RUNTIME.mkdir(); CACHE.mkdir(); HF_HOME.mkdir(); manifest = RUNTIME/'runtime.json'
boot = ['kaggle-vllm', 'bootstrap', '--strict', '--staged', str(RUNTIME/'staged'), '--overlay', str(RUNTIME/'overlay'), '--cache', str(CACHE), '--manifest', str(manifest)]
subprocess.run(boot + ['--dry-run'], check=True); subprocess.run(boot, check=True)
runtime = json.loads(manifest.read_text()); RUN_ENV = dict(os.environ); RUN_ENV.update(runtime['runtime_environment']); RUN_ENV['PYTHONPATH'] = str(SOURCE/'src') + os.pathsep + str(SOURCE) + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
print(json.dumps({'source_commit': head, 'batch_id': batch_id, 'repetition': batch['repetition'], 'ordered_shards': batch['ordered_shard_ids'], 'disk_total_bytes': disk.total, 'disk_free_bytes': disk.free, 'disk_reserve_bytes': reserve, 'token_present_not_printed': True}, indent=2))

In [ ]:
command = [sys.executable, str(SOURCE/'scripts/kaggle_m4_execute_batch.py'), '--repository', str(SOURCE), '--output-root', str(OUTPUT_ROOT), '--runtime', str(manifest), '--batch-id', batch_id, '--batch-plan', 'research/M4_BATCH_EXECUTION_PLAN_V2.json', '--source-identity', EXPECTED_SOURCE_COMMIT, '--batch-notebook-source-digest', BATCH_NOTEBOOK_SOURCE_DIGEST]
completed = subprocess.run(command, cwd=SOURCE, env=RUN_ENV, check=False)
outer = WORK/f'm4-batch-{batch_id}.zip'; print(json.dumps({'batch_runner_returncode': completed.returncode, 'outer_zip': str(outer)}, indent=2))
if not outer.is_file(): raise RuntimeError('batch runner did not preserve an outer archive')
outer_sha256 = hashlib.sha256(outer.read_bytes()).hexdigest(); print(json.dumps({'outer_zip': str(outer), 'outer_zip_sha256': outer_sha256, 'runtime_json': str(manifest)}, indent=2))
if completed.returncode not in {0, 2}: raise RuntimeError(f'unexpected batch runner return code {completed.returncode}; preserve all outputs for review')

Download the printed outer ZIP, this executed notebook, and `/kaggle/working/kaggle-vllm-runtime/runtime.json`. Do not download Hugging Face caches, model weights, the native wheel, or the historical Qwen TP=2 sharded-state archive. A completed batch is still only a set of canonical candidates until local `ingest-m4-batch` succeeds.